In [1]:
# conda env 'at3d_pure'

from collections import OrderedDict
import at3d
import xarray as xr
import numpy as np

In [2]:
## INPUT
aerosol_file = '../intermediate_data/AEROSOL_MISR_21_0p250.nc'
atmos_file = '../intermediate_data/ATMOS_S07_RTG_1400.nc'

## OUTPUT
out_file = '../intermediate_data/MIXTURE_default_0p250.nc'

In [3]:
# wavelength = 0.67

aerosol = xr.load_dataset(aerosol_file)
atmos = xr.load_dataset(atmos_file)

assert np.array_equal(aerosol.x.data, atmos.x.data)
assert np.array_equal(aerosol.y.data, atmos.y.data)
assert np.array_equal(aerosol.z.data, atmos.z.data)

In [4]:
aerosol

<xarray.Dataset>
Dimensions:        (x: 65, y: 65, z: 16, num_micro: 1, stokes_index: 6,
                    legendre_index: 21, table_index: 1)
Coordinates:
  * x              (x) float64 0.0 0.275 0.55 0.825 ... 16.78 17.05 17.33 17.6
  * y              (y) float64 0.0 0.275 0.55 0.825 ... 16.78 17.05 17.33 17.6
  * z              (z) float64 0.0 0.1 0.2 0.3 0.39 0.4 ... 1.0 1.1 1.2 1.3 1.4
    table_index    (num_micro, x, y, z) int32 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1
Dimensions without coordinates: num_micro, stokes_index, legendre_index
Data variables:
    delx           float64 0.275
    dely           float64 0.275
    z_ext          (z) float64 0.1258 0.1197 0.1139 ... 0.06907 0.0657 2.528
    ssalb          (x, y, z) float64 1.0 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0 1.0 1.0
    extinction     (x, y, z) float64 0.1258 0.1197 0.1139 ... 0.0657 2.528
    phase_weights  (num_micro, x, y, z) float64 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0
    legcoef        (stokes_index, legendre_index, table_index) float64 1.0 .....
Attributes:
    mixture:        21
    band:           0.67175
    leg_residuals:  0.00037784383

In [5]:
background_optical_scatterers = OrderedDict()
merged_properties = at3d.medium.mix_optical_properties(
    atmos,
    aerosol,
    asymmetry_tol=0.02,
    maxnphase=40
)

ones = np.ones((atmos.dims['x'], atmos.dims['y'], atmos.dims['z']))
if((np.array_equal(atmos.ssalb.data, ones)) & (np.array_equal(aerosol.ssalb.data, ones))):
    merged_properties['ssalb'] = (['x', 'y', 'z'], ones)

merged_properties.to_netcdf(out_file)